In [ ]:
from pathlib import Path

import duckdb

ROOT = Path.cwd().parent              # EDA/ -> Toy_Storey_Project/
DB_FILE = ROOT / "data" / "toystore.duckdb"

con = duckdb.connect(str(DB_FILE), read_only=True)
con.sql("show all tables").show()

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
products = pd.read_csv(ROOT / "data" / "products.csv")

transactions = []
for file in sorted((ROOT / "data" / "sales").glob("sales_*.jsonl")):
    with open(file, encoding="utf-8") as f:
        transactions.extend(json.loads(line) for line in f if line.strip())

receipts = pd.json_normalize(transactions).drop(columns="items")
receipts["date"] = pd.to_datetime(receipts["transaction_timestamp"].str[:10])

lines = pd.json_normalize(transactions, record_path="items", meta=["transaction_timestamp"])
lines["date"] = pd.to_datetime(lines["transaction_timestamp"].str[:10])
lines = lines.merge(products[["product_id", "product_name", "category", "popularity_weight"]], on="product_id")

Does the product sell according to weight?

In [ ]:
per_product = (
    lines.groupby(["product_name", "popularity_weight"])["quantity"].sum()
    .reset_index().sort_values("quantity", ascending=False)
)
per_product

Receipt per day over the year

In [ ]:
per_day = receipts.groupby("date").size()
per_day.plot(figsize=(14, 4), title="Transactions per day")

Black Friday and payday in numbers

In [ ]:
def avg(mask):
    return round(per_day[mask].mean(), 1)

idx = per_day.index
normal = (idx.month.isin([3, 4, 5, 9, 10])) & (idx.day < 25)

print("Normal day:      ", avg(normal))
print("Payday 25-28:    ", avg((idx.month.isin([3, 4, 5, 9, 10])) & (idx.day >= 25) & (idx.day <= 28)))
print("Black Friday:    ", per_day.get(pd.Timestamp("2025-11-28")))
print("December 1-23:   ", avg((idx.month == 12) & (idx.day <= 23)))

Season per category

In [ ]:
season = (
    lines.assign(month=lines["date"].dt.month)
    .groupby(["month", "category"])["quantity"].sum()
    .unstack()
)
share = season.div(season.sum(axis=1), axis=0)   # share of the month's sales
share[["Fordon", "Pussel", "Sällskapsspel"]].plot(figsize=(12, 4), title="Share of sales per month")

Check if inventory ever becomes negative

In [ ]:
inventory = []
for file in sorted((ROOT / "data" / "inventory").glob("inventory_*.jsonl")):
    with open(file, encoding="utf-8") as f:
        inventory.extend(json.loads(line) for line in f if line.strip())

items = pd.json_normalize(inventory, record_path="items")
print("Min on hand:", items["quantity_on_hand"].min())
print(items["stock_status"].value_counts())